In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np

import plotly
from plotly.graph_objects import Scatter
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=False)
import plotly.express as px

import cosmosdr

import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc
from cosmosdr.plotting import create_base_figure

try:
    sdr.close()
except:
    pass

# Interpreting aircraft signals

In [ ]:
ADSB_FREQUENCY = 1090e6
# Total number of bits in a single ADSB message
ADSB_BITS = 112
# One microsecond timeslot for one bit, 'on' if signal is within the first half of this slot
ADSB_SLOT_LENGTH = 1 / 1e6

center_freq = ADSB_FREQUENCY
# reccomended upper limit of sample rate. Fast enough to oversample
sample_rate = 2e6
n_reads = 16
n_samples = 4096*8  # more samples seems to just make it noisy

In [ ]:
1 / sample_rate

In [ ]:
ADSB_SLOT_LENGTH / 2

In [ ]:
# Assert that we are sampling at the same rate as the signal comms (so we can plot easily later
assert (1 / sample_rate) == ((ADSB_SLOT_LENGTH / 2))

In [ ]:
# Start up the SDR connection
try:
    sdr.close()
except:
    pass

sdr = s_acq.get_sdr(center_freq=center_freq, sample_rate=sample_rate)

s = s_acq.acquire_signal(sdr, n_reads=n_reads, n_samples=n_samples)


In [ ]:
s_df = pd.DataFrame(np.abs(s)).T

# Plot all the reads, see where there were peaks

In [ ]:
px.line(s_df.rolling(16).max()[::64])

### Plot the samples of the best candidate read
Given that this capture had the highest peak magnitude within it, it likely included an ASDB pulse

In [ ]:
# Grab the column with the highest individual peak
signal_col = s_df[s_df.max().idxmax()]

In [ ]:
fig = px.bar(signal_col)

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

# fig.update_layout(bargap=0,
#                   bargroupgap = 0,
#                  )


# Plot the on/off signalling

- If the peak is one colour, then the signal was in the first half of a slot
- If the peak is the other,  then the signal was in the second half of a slot

On or off depends on the start point, can't be predicted assessed ahead of time

In [ ]:
evens_indexer = signal_col.index % 2 == 0

In [ ]:
# Split the first half/second half into separate columns so they can be plotted differently
even = signal_col.reindex(signal_col.index[evens_indexer])
odd  = signal_col.reindex(signal_col.index[~evens_indexer])
even.name="even"
odd.name="odd"

# Plot the strongest signal to highlight 1s and 0s
- Within each 1us (microsecond, millionth of a second), there are two halves to the 'frame'
- If there is a signal peak within the first half, this is a 1
- If there is a signal peak within the second half, this is a 0

In [ ]:
fig = px.bar(pd.concat([even, odd], axis=1))

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))